# Sprint 11 · Modelado de Datos y Dashboards con Power BI: Caso Megaline 📊**Tema:** Modelado de datos, medidas avanzadas y dashboards interactivos en Power BI.> 🚀 En esta sesión, transformaremos datos crudos de **Megaline** en un dashboard interactivo y revelador. Aprenderás a construir un modelo de datos robusto, escribirás tus primeras fórmulas en DAX y diseñarás un panel que comunica ideas de negocio de forma efectiva.**Programa:** Data Analytics · **Sprint:** 11 · **Modalidad:** Práctico-Teórico

## 🎯 Objetivos de la sesiónAl finalizar esta clase, serás capaz de:1.  Conectar múltiples fuentes de datos en Power BI.2.  Construir un **modelo en estrella** para asegurar la integridad de tu análisis.3.  Crear **columnas calculadas y medidas** usando el lenguaje DAX.4.  Aplicar la función `CALCULATE` para modificar el contexto de tus cálculos.5.  Diseñar un **dashboard interactivo** que responda a preguntas de negocio clave.

## ⏰ Agenda Sugerida| Tiempo | Bloque | Contenido ||---:|---|---|| 0–10 | Bienvenida y Objetivos | Introducción al caso de Megaline y metas de la sesión. || 10–30 | Modelo de Datos | Conectar las 5 tablas de Megaline y construir el modelo en estrella. || 30–60 | Magia con DAX | Creación de medidas para ingresos, costos y uso de servicios. || 60–85 | Diseño del Dashboard | Ensamblar los visuales para contar una historia con los datos. || 85–95 | Interactividad | Agregar filtros y segmentadores para explorar los datos. || 95–100| Cierre y Conclusiones | Repaso de los aprendizajes y próximos pasos. |

## 📈 El Desafío: Analizando los Datos de MegalineMegaline, una empresa de telecomunicaciones, quiere entender mejor el comportamiento de sus clientes. Necesitan saber qué plan es más rentable y cómo se distribuye el consumo de llamadas, mensajes y datos. ¡Tu misión es ayudarlos a visualizar esta información!### Nuestras Fuentes de DatosTrabajaremos con 5 tablas que describen el negocio:| Tabla | Descripción | Rol en el modelo ||---|---|---|| **users** | Información demográfica de los clientes. | Dimensión || **plans** | Detalles de las tarifas `Surf` y `Ultimate`. | Dimensión || **calls** | Registro de todas las llamadas realizadas. | Tabla de Hechos || **messages**| Registro de todos los SMS enviados. | Tabla de Hechos || **internet**| Registro del consumo de datos en cada sesión. | Tabla de Hechos |> 🤔 **Reflexión:** ¿Por qué tenemos tres tablas de hechos? Porque los eventos (llamadas, mensajes, datos) ocurren de forma independiente y tienen distinta granularidad. Nuestro modelo deberá reflejar esto.

## 🛠️ Paso 1: Construyendo el Modelo en EstrellaUn buen dashboard empieza con un buen modelo. Un **modelo en estrella** es como el chasis de un coche: le da estructura y solidez a todo lo que construyas encima.**Tu Misión:**1.  **Conectar los datos:** En Power BI, usa `Obtener datos` para cargar las 5 tablas de Megaline (pueden ser CSV o de una base de datos).2.  **Ir a la Vista de Modelo:** Aquí es donde ocurre la magia.3.  **Crear relaciones:** Power BI podría crear algunas relaciones automáticamente. ¡Verifícalas! Arrastra y suelta las columnas para crear las siguientes conexiones:    *   `users[user_id]` → `calls[user_id]` (Uno a Varios)    *   `users[user_id]` → `messages[user_id]` (Uno a Varios)    *   `users[user_id]` → `internet[user_id]` (Uno a Varios)    *   `users[plan]` → `plans[plan_name]` (Uno a Varios)El esquema debería verse así, con `users` y `plans` como dimensiones centrales que describen los hechos.```         ┌─────────┐         │  plans  │         └────┬────┘              │ plan_name┌────────┐    ││  users │────┼────> (calls, messages, internet)└────────┘ user_id   (Tablas de Hechos)```> ✅ **Tip:** Asegúrate de que la dirección del filtro sea siempre **desde la dimensión hacia la tabla de hechos** (la flecha apunta hacia la tabla de hechos).

## 🧠 Paso 2: ¡Hablando el Idioma de los Datos con DAX!**DAX (Data Analysis Expressions)** es el lenguaje de fórmulas de Power BI. Al principio puede parecer intimidante, pero es pura lógica.> **Regla de Oro:** Prefiere siempre **Medidas** sobre **Columnas Calculadas**. Las medidas son dinámicas y se calculan según los filtros de tu visual, ¡lo que las hace súper potentes y eficientes!

### 2.1 El Prerrequisito: La Tabla CalendarioPara analizar datos por mes, trimestre o año, necesitas una tabla calendario. Esta es la columna vertebral de todo análisis temporal.**Tu Misión:**1.  Ve a la `Vista de datos`.2.  En la cinta `Herramientas de tablas`, haz clic en `Nueva tabla`.3.  Pega el siguiente código DAX para crear una tabla calendario dinámica:

In [ ]:
Calendario =ADDCOLUMNS (    CALENDAR ( MIN ( internet[session_date] ), MAX ( internet[session_date] ) ),    "Año", YEAR ( [Date] ),    "Mes", FORMAT ( [Date], "YYYY-MM" ),    "Mes Num", MONTH ( [Date] ))

4.  **Importante:** Ve a la `Vista de modelo` y crea una relación entre `Calendario[Date]` y `internet[session_date]`, `calls[call_date]`, y `messages[message_date]`.5.  Finalmente, haz clic derecho en la tabla `Calendario` y selecciona `Marcar como tabla de fechas`.

### 2.2 El Gran Desafío: Calcular la Factura Mensual por UsuarioAquí es donde demostramos el poder de DAX. Tu pregunta es clave: ¿cómo calculamos la factura de cada usuario, cada mes, basándonos en su consumo y su plan? Lo haremos con medidas, no creando tablas físicas.#### Paso A: Medidas de Consumo BásicoPrimero, las medidas simples que agregan el total.

In [ ]:
// 1. Total de minutos de llamadaTotal Minutos = SUM(calls[duration])// 2. Total de mensajes enviadosTotal SMS = COUNT(messages[id])// 3. Total de datos consumidos (en GB)Total GB Usados = SUM(internet[mb_used]) / 1024

#### Paso B: Medidas para Costos AdicionalesAhora, creamos la lógica para calcular el costo de cada excedente. Estas medidas funcionarán dinámicamente para cualquier usuario/mes seleccionado.

In [ ]:
// Costo por minutos extraCosto Minutos Extra =VAR MinutosUsados = [Total Minutos]VAR MinutosIncluidos = SELECTEDVALUE(plans[minutes_included])VAR MinutosExtra = MAX(0, MinutosUsados - MinutosIncluidos)VAR CostoPorMinutoExtra = SELECTEDVALUE(plans[usd_per_minute])RETURN MinutosExtra * CostoPorMinutoExtra// Costo por SMS extraCosto SMS Extra =VAR SMSUsados = [Total SMS]VAR SMSIncluidos = SELECTEDVALUE(plans[messages_included])VAR SMSExtra = MAX(0, SMSUsados - SMSIncluidos)VAR CostoPorSMSExtra = SELECTEDVALUE(plans[usd_per_message])RETURN SMSExtra * CostoPorSMSExtra// Costo por GB extraCosto GB Extra =VAR GBUsados = [Total GB Usados]VAR GBIncluidos = SELECTEDVALUE(plans[mb_per_month_included]) / 1024VAR GBExtra = MAX(0, GBUsados - GBIncluidos)VAR CostoPorGBExtra = SELECTEDVALUE(plans[usd_per_gb])RETURN CEILING(GBExtra, 1) * CostoPorGBExtra// Nota: Usamos CEILING para redondear hacia arriba el GB extra, como pide Megaline.

#### Paso C: La Medida Maestra de Facturación TotalEsta es la medida que une todo. Usaremos la función `SUMX`, que es un "iterador". Recorrerá una tabla virtual (en este caso, cada combinación de usuario y mes) y sumará el resultado de nuestros cálculos.

In [ ]:
Facturacion Total =SUMX(    SUMMARIZE(users, users[user_id], Calendario[Mes]),    // La expresión siguiente se calcula para cada usuario y mes    VAR TarifaBase = SELECTEDVALUE(plans[usd_monthly_fee])    VAR CostoMinutos = [Costo Minutos Extra]    VAR CostoSMS = [Costo SMS Extra]    VAR CostoDatos = [Costo GB Extra]    RETURN TarifaBase + CostoMinutos + CostoSMS + CostoDatos)

> 🤯 **¡Wow! ¿Qué hace esta medida?**> `SUMMARIZE` crea una tabla virtual con cada `user_id` y `Mes`.> `SUMX` itera sobre cada fila de esa tabla virtual y calcula la factura total (tarifa base + todos los extras).> Finalmente, suma los resultados de todas las filas para darte la facturación total del período que estés viendo en tu gráfico. ¡Así es como Power BI maneja cálculos complejos a nivel de fila de forma dinámica!

## 🎨 Paso 3: Creando un Dashboard para la GerenciaUn buen dashboard cuenta una historia. La gerencia de Megaline no quiere ver tablas y números, quiere respuestas.**Preguntas a Responder:***   ¿Cuál es el ingreso total por plan? (Ahora puedes usar tu medida `[Facturacion Total]`)*   ¿Qué plan tiene más usuarios?*   ¿Cómo es el consumo promedio (minutos, SMS, datos) por usuario para cada plan?*   ¿Hay usuarios que exceden consistentemente los límites de su plan? (¡Tus medidas de costo extra responden esto!)**Tu Misión:**1.  **KPIs Principales:** Usa tarjetas (`Card`) para mostrar las métricas más importantes:    *   `[Facturacion Total]`    *   Número de Clientes (puedes crear una medida `COUNTROWS(users)`)    *   `[Costo GB Extra]` (para ver cuánto se gana por datos adicionales)2.  **Gráficos de Comparación:**    *   **Gráfico de barras agrupadas:** Compara `[Facturacion Total]` por `plan`.    *   **Gráfico de barras apiladas:** Muestra la distribución de usuarios por `ciudad` y `plan`.3.  **Visual de Tendencia:**    *   **Gráfico de líneas:** Muestra la evolución de `[Facturacion Total]` a lo largo del tiempo, usando tu `Calendario[Mes]`.4.  **Añade Interactividad:**    *   Usa un **Segmentador de datos (Slicer)** para filtrar todo el dashboard por `plan` o `ciudad`. ¡Verás cómo todos los gráficos se actualizan al instante!> ✨ **Toque final:** Usa colores consistentes. Por ejemplo, asigna siempre el mismo color al plan `Surf` y otro al plan `Ultimate` en todos los gráficos. Esto hace que tu dashboard sea más fácil de leer y entender.

## 🎉 Cierre y Próximos Pasos¡Felicidades! Has pasado de datos crudos a un dashboard funcional y con propósito. Y no solo eso, has construido un motor de cálculo de facturación complejo y robusto.**Hoy aprendiste a:***   Estructurar un modelo de datos en estrella y la importancia de una **tabla calendario**.*   Escribir fórmulas DAX para manejar lógica de negocio compleja (cálculo de excedentes).*   Usar **iteradores como `SUMX`** para realizar cálculos a nivel de fila dentro de una medida.*   Diseñar un dashboard que responde preguntas de negocio profundas.**Tu próximo desafío:** ¿Puedes crear una medida que calcule el "Coste de Adquisición vs. Valor del Cliente (LTV)"? ¡Ya tienes la facturación mensual, el primer paso para calcular el LTV!